# Forensic Multi-Domain YOLO Rerun — Full Training Pipeline

**Paper:** *Merging Multi-Domain Datasets for Generalized Forensic Evidence Detection*

This notebook runs the complete experimental pipeline:
- 4 model configs (YOLOv8-m baseline/enhanced, YOLO11-m baseline/enhanced)
- 3 random seeds each (42, 123, 7) → 12 total training runs
- 150 epochs per run
- Cross-domain evaluation (crime-scene vs smart-device classes)
- ONNX export for edge deployment analysis
- Model profiling (params, GFLOPs, latency)

**Hardware:** Google Colab Pro — L4 GPU (or better)
**Estimated time:** ~18–24 hours total

## 0. Setup and Installation

In [ ]:
# Install dependencies
!pip install ultralytics --quiet
!pip install onnx onnxruntime-gpu --quiet

In [ ]:
import os
import json
import yaml
import shutil
import time
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration — Edit These Paths

In [ ]:
# =============================================================================
# CONFIGURATION — PATHS VERIFIED FROM YOUR data.yaml
# =============================================================================

# The dataset data.yaml already uses absolute Colab paths:
#   path:  /content/drive/MyDrive/ForensicsDataset/merged_dataset
#   train: /content/drive/MyDrive/ForensicsDataset/merged_dataset/images/train
#   val:   /content/drive/MyDrive/ForensicsDataset/merged_dataset/images/val
#   test:  /content/drive/MyDrive/ForensicsDataset/merged_dataset/images/test

# Dataset YAML (absolute path on Colab after Drive mount)
DATASET_YAML = "/content/drive/MyDrive/ForensicsDataset/merged_dataset/data.yaml"

# Where to save all experiment outputs
OUTPUT_ROOT = "/content/drive/MyDrive/ForensicsDataset/rerun_experiments_v2"

# Folder containing the corrected YAML config files
# >>> Upload yolov8m_enhanced.yaml and yolo11m_enhanced.yaml here <<<
CONFIG_DIR = "/content/drive/MyDrive/ForensicsDataset/configs_v2"

# =============================================================================
# EXPERIMENT MATRIX
# =============================================================================

MODELS = {
    "yolov8m_baseline": {
        # Baseline: load from official .pt (architecture + COCO pretrained weights)
        "source": "yolov8m.pt",
        "is_custom_yaml": False,
        "expected_depth": 0.67,
        "expected_width": 0.75,
        "expected_max_ch": 768,
        "family": "yolov8",
    },
    "yolov8m_enhanced": {
        # Enhanced: custom YAML with same m-scale multipliers + added layers
        "source": os.path.join(CONFIG_DIR, "yolov8m_enhanced.yaml"),
        "is_custom_yaml": True,
        "expected_depth": 0.67,
        "expected_width": 0.75,
        "expected_max_ch": 768,
        "family": "yolov8",
    },
    "yolo11m_baseline": {
        # Baseline: load from official .pt
        "source": "yolo11m.pt",
        "is_custom_yaml": False,
        "expected_depth": 0.50,
        "expected_width": 1.00,
        "expected_max_ch": 512,
        "family": "yolo11",
    },
    "yolo11m_enhanced": {
        # Enhanced: hardcoded YAML (no multipliers — avoids scale parsing bug)
        "source": os.path.join(CONFIG_DIR, "yolo11m_enhanced.yaml"),
        "is_custom_yaml": True,
        "hardcoded": True,  # channels pre-scaled, no multiplier check needed
        "family": "yolo11",
    },
}

SEEDS = [42, 123, 7]
EPOCHS = 150          # Increased from 100 for better convergence
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = 0            # GPU index

## 2. Verify Setup
This cell validates that all configs exist, the dataset is reachable, and — critically — that every YAML has the correct medium-scale multipliers.

In [ ]:
def verify_setup():
    """Check configs, dataset, and scale multipliers before training."""
    print("=" * 60)
    print("SETUP VERIFICATION")
    print("=" * 60)

    # Check dataset
    assert os.path.exists(DATASET_YAML), f"Dataset YAML not found: {DATASET_YAML}"
    with open(DATASET_YAML, "r") as f:
        ds = yaml.safe_load(f)
    print(f"[OK] Dataset: {ds.get('nc', '?')} classes, path: {ds.get('path', '?')}")

    # Check each model config
    for name, cfg in MODELS.items():
        if cfg["is_custom_yaml"]:
            assert os.path.exists(cfg["source"]), f"Config not found: {cfg['source']}"
            with open(cfg["source"], "r") as f:
                y = yaml.safe_load(f)
            nc = y.get("nc", "?")

            if cfg.get("hardcoded"):
                # Hardcoded YAML — no multipliers to check, just verify nc
                print(f"[OK] {name}: hardcoded architecture, nc={nc}")
            else:
                # Check scale multipliers
                dm = y.get("depth_multiple")
                wm = y.get("width_multiple")
                mc = y.get("max_channels")
                print(f"[CHECK] {name}: depth={dm}, width={wm}, max_ch={mc}, nc={nc}")
                assert dm == cfg["expected_depth"], \
                    f"WRONG depth for {name}: {dm} (expected {cfg['expected_depth']})"
                assert wm == cfg["expected_width"], \
                    f"WRONG width for {name}: {wm} (expected {cfg['expected_width']})"
                assert mc == cfg["expected_max_ch"], \
                    f"WRONG max_ch for {name}: {mc} (expected {cfg['expected_max_ch']})"
                print(f"  [OK] Scale multipliers verified for {cfg['family']}-m")
        else:
            print(f"[OK] {name}: using official {cfg['source']}")

    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    print(f"[OK] Output dir: {OUTPUT_ROOT}")
    print("=" * 60)
    print("ALL CHECKS PASSED")

verify_setup()

## 3. Training — All Models × All Seeds
This runs all 12 experiments sequentially. Each takes ~1.5–2 hours on an L4 GPU.

In [ ]:
def train_single_run(model_name, model_cfg, seed, run_id):
    """Train one model with one seed. Train locally, then copy to Drive."""

    # Train to LOCAL directory (avoids ultralytics project name validation bug)
    local_project = f"/content/runs/{model_name}"
    local_name = f"seed_{seed}"

    # Final destination on Drive
    drive_dir = os.path.join(OUTPUT_ROOT, model_name, f"seed_{seed}")

    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name} | seed={seed} | run {run_id}/12")
    print(f"Local: {local_project}/{local_name}")
    print(f"Drive: {drive_dir}")
    print(f"{'='*60}\n")

    # Build model
    model = YOLO(model_cfg["source"])

    n_params = sum(p.numel() for p in model.model.parameters())
    print(f"  Loaded: {model_cfg['source']}")
    print(f"  Parameters: {n_params/1e6:.2f}M")

    # Train to local path
    results = model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        project=local_project,
        name=local_name,
        exist_ok=True,
        seed=seed,
        pretrained=True,
        # Optimizer (consistent across all runs)
        optimizer="SGD",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        # Augmentation (default YOLO pipeline)
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        # Logging
        verbose=True,
        plots=True,
        save=True,
        save_period=50,
    )

    # Copy results to Drive
    local_results = os.path.join(local_project, local_name)
    os.makedirs(drive_dir, exist_ok=True)

    # Copy the train folder contents to Drive
    drive_train_dir = os.path.join(drive_dir, "train")
    if os.path.exists(drive_train_dir):
        shutil.rmtree(drive_train_dir)
    shutil.copytree(local_results, drive_train_dir)
    print(f"\n  Copied results to: {drive_train_dir}")

    # Save run metadata
    meta = {
        "model_name": model_name,
        "seed": seed,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "source": model_cfg["source"],
        "family": model_cfg["family"],
        "timestamp": datetime.now().isoformat(),
    }
    with open(os.path.join(drive_dir, "run_metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    # Clean up local to save disk
    shutil.rmtree(local_results, ignore_errors=True)

    return drive_dir

In [ ]:
def run_all_training():
    """Execute all 12 training runs."""
    all_runs = []
    run_id = 0

    for model_name, model_cfg in MODELS.items():
        for seed in SEEDS:
            run_id += 1
            print(f"\n>>> Starting run {run_id}/12: {model_name}, seed={seed}")
            try:
                run_dir = train_single_run(model_name, model_cfg, seed, run_id)
                all_runs.append({
                    "model": model_name, "seed": seed,
                    "dir": run_dir, "status": "success"
                })
            except Exception as e:
                print(f"!!! FAILED: {model_name} seed={seed}: {e}")
                all_runs.append({
                    "model": model_name, "seed": seed,
                    "dir": None, "status": f"failed: {e}"
                })

    # Save run summary
    summary_path = os.path.join(OUTPUT_ROOT, "training_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_runs, f, indent=2)
    print(f"\n\nTraining summary saved: {summary_path}")
    return all_runs

# ===== LAUNCH ALL TRAINING =====
all_runs = run_all_training()

## 4. Evaluation — Combined + Cross-Domain
Evaluate all trained models on the test set, with per-domain mAP breakdown.

In [ ]:
def evaluate_model(weights_path, data_yaml, split="test", device=0):
    """Run evaluation and return metrics dict."""
    model = YOLO(weights_path)
    results = model.val(
        data=data_yaml,
        split=split,
        device=device,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        verbose=True,
        plots=True,
        save_json=True,
    )
    metrics = {
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
        "mAP50": float(results.box.map50),
        "mAP50_95": float(results.box.map),
    }
    # Per-class AP if available
    try:
        class_names = list(results.names.values())
        ap50_vals = results.box.ap50.tolist()
        metrics["per_class_ap50"] = dict(zip(class_names, ap50_vals))
    except Exception:
        metrics["per_class_ap50"] = {}
    return metrics, results

In [ ]:
def run_full_evaluation():
    """Evaluate all trained models on the test set."""

    # Domain class ranges: crime-scene = 0-37, smart-device = 38-47
    cs_indices = set(range(0, 38))
    sd_indices = set(range(38, 48))

    results_all = []

    for model_name in MODELS.keys():
        for seed in SEEDS:
            weights = os.path.join(
                OUTPUT_ROOT, model_name, f"seed_{seed}", "train", "weights", "best.pt"
            )
            if not os.path.exists(weights):
                print(f"[SKIP] Weights not found: {weights}")
                continue

            print(f"\n--- Evaluating {model_name} seed={seed} ---")
            metrics, raw = evaluate_model(weights, DATASET_YAML, split="test")

            # Split per-class AP by domain
            per_class = metrics.get("per_class_ap50", {})
            class_list = list(per_class.keys())
            cs_aps = [per_class[c] for i, c in enumerate(class_list) if i in cs_indices]
            sd_aps = [per_class[c] for i, c in enumerate(class_list) if i in sd_indices]

            row = {
                "model": model_name,
                "seed": seed,
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "mAP50": metrics["mAP50"],
                "mAP50_95": metrics["mAP50_95"],
                "mAP50_crime_scene": float(np.mean(cs_aps)) if cs_aps else None,
                "mAP50_smart_device": float(np.mean(sd_aps)) if sd_aps else None,
            }
            results_all.append(row)

    df = pd.DataFrame(results_all)
    df.to_csv(os.path.join(OUTPUT_ROOT, "all_evaluation_results.csv"), index=False)

    # Mean ± std per model
    summary = df.groupby("model").agg(
        precision_mean=("precision", "mean"), precision_std=("precision", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        mAP50_mean=("mAP50", "mean"), mAP50_std=("mAP50", "std"),
        mAP50_95_mean=("mAP50_95", "mean"), mAP50_95_std=("mAP50_95", "std"),
        mAP50_cs_mean=("mAP50_crime_scene", "mean"), mAP50_cs_std=("mAP50_crime_scene", "std"),
        mAP50_sd_mean=("mAP50_smart_device", "mean"), mAP50_sd_std=("mAP50_smart_device", "std"),
    ).round(4)

    summary.to_csv(os.path.join(OUTPUT_ROOT, "model_summary_mean_std.csv"))
    print("\n" + "=" * 80)
    print("EVALUATION SUMMARY (mean +/- std across 3 seeds)")
    print("=" * 80)
    print(summary.to_string())
    return df, summary

# ===== RUN EVALUATION =====
eval_df, eval_summary = run_full_evaluation()

## 5. Model Profiling — GFLOPs, Parameters, Inference Latency

In [ ]:
import torch

def profile_all_models():
    """Get parameter count, GFLOPs, and inference speed for each architecture."""
    profiles = []

    for model_name, model_cfg in MODELS.items():
        print(f"\n--- Profiling {model_name} ---")
        model = YOLO(model_cfg["source"])

        # Get model summary (logs params and GFLOPs)
        info = model.info(verbose=True, detailed=False)

        # Measure inference latency
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).cuda()
        pytorch_model = model.model.cuda().eval()

        # Warmup
        for _ in range(10):
            with torch.no_grad():
                _ = pytorch_model(dummy)

        torch.cuda.synchronize()
        times = []
        for _ in range(100):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = pytorch_model(dummy)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000)

        profiles.append({
            "model": model_name,
            "parameters_M": round(sum(p.numel() for p in pytorch_model.parameters()) / 1e6, 2),
            "GFLOPs": info[1] if isinstance(info, tuple) else "see_logs",
            "inference_ms_mean": round(np.mean(times), 2),
            "inference_ms_std": round(np.std(times), 2),
        })

    df_profiles = pd.DataFrame(profiles)
    df_profiles.to_csv(os.path.join(OUTPUT_ROOT, "model_profiles.csv"), index=False)
    print("\n" + df_profiles.to_string(index=False))
    return df_profiles

profiles_df = profile_all_models()

## 6. ONNX Export for Edge Deployment Analysis

In [ ]:
def export_onnx_all():
    """Export best weights to ONNX and measure file size."""
    exports = []

    for model_name in MODELS.keys():
        weights = os.path.join(
            OUTPUT_ROOT, model_name, "seed_42", "train", "weights", "best.pt"
        )
        if not os.path.exists(weights):
            print(f"[SKIP] {model_name}: weights not found")
            continue

        print(f"\n--- Exporting {model_name} to ONNX ---")
        model = YOLO(weights)
        export_path = model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)

        file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
        exports.append({
            "model": model_name,
            "onnx_path": str(export_path),
            "onnx_size_MB": round(file_size_mb, 2),
        })

    df_exports = pd.DataFrame(exports)
    df_exports.to_csv(os.path.join(OUTPUT_ROOT, "onnx_exports.csv"), index=False)
    print("\n" + df_exports.to_string(index=False))
    return df_exports

exports_df = export_onnx_all()

## 7. Summary

In [ ]:
print(f"""
{'='*70}
PIPELINE COMPLETE
{'='*70}
Outputs saved to: {OUTPUT_ROOT}

Files generated:
  training_summary.json        — run status for all 12 training runs
  all_evaluation_results.csv   — per-run metrics including cross-domain
  model_summary_mean_std.csv   — mean +/- std per model (for paper tables)
  model_profiles.csv           — params, GFLOPs, latency (for paper tables)
  onnx_exports.csv             — ONNX file sizes (for deployment section)

Per-run folders contain:
  train/weights/best.pt & last.pt
  Confusion matrices, PR curves, F1 curves
  Training loss curves (results.csv)

Next step:
  Open 02_analysis_and_plots.ipynb to generate paper-ready figures.
{'='*70}
""")